In [ ]:
import requests
import json

# URL 문자열
url ='https://apihub.kma.go.kr/api/typ01/url/kma_sfctm2.php?tm=202211300900&stn=0&help=1&authKey=6XEt4XIFQJCxLeFyBYCQjg'

# GET 요청
response = requests.get(url)

# 응답을 JSON 형태로 변환
json_response = response.text

print(json_response)

In [ ]:
import requests
import pandas as pd

# API 설정
AUTH_KEY = 'YOUR_AUTH_KEY_HERE'  # ← 본인의 인증키로 변경!
BASE_URL = 'https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php'

print("=" * 80)
print("기상청 API 응답 구조 확인")
print("=" * 80)

# 테스트 파라미터 (2021년 1월 1일 데이터)
test_params = {
    'tm1': '202101010000',
    'tm2': '202101011000',  # 10시간만 테스트
    'stn': '108',  # 서울
    'authKey': '6XEt4XIFQJCxLeFyBYCQjg',
    'help': 1  # help=1로 컬럼 정보 확인
}

print("\n[Step 1] help=1로 API 구조 확인")
print(f"요청 URL: {BASE_URL}")
print(f"파라미터: {test_params}")
print("-" * 80)

try:
    response = requests.get(BASE_URL, params=test_params, timeout=30)
    print(f"\n응답 상태 코드: {response.status_code}")
    
    if response.status_code == 200:
        print("\n✓ API 연결 성공!")
        print("\n[API 응답 내용]")
        print("-" * 80)
        print(response.text[:2000])  # 처음 2000자만 출력
        print("-" * 80)
        
        # 응답을 줄 단위로 분리
        lines = response.text.strip().split('\n')
        print(f"\n총 {len(lines)}줄의 응답")
        
        if len(lines) > 0:
            print(f"\n[첫 번째 줄 - 헤더 또는 설명]")
            print(lines[0])
            
            # 헤더 분석
            first_line_parts = lines[0].split()
            print(f"\n컬럼 개수: {len(first_line_parts)}")
            print(f"컬럼 목록: {first_line_parts}")
        
    else:
        print(f"\n✗ API 오류: {response.status_code}")
        print(f"응답 내용: {response.text}")
        
except Exception as e:
    print(f"\n✗ 요청 실패: {e}")

# Step 2: help=0으로 실제 데이터 확인
print("\n\n" + "=" * 80)
print("[Step 2] help=0으로 실제 데이터 확인")
print("=" * 80)

test_params['help'] = 0

try:
    response = requests.get(BASE_URL, params=test_params, timeout=30)
    print(f"\n응답 상태 코드: {response.status_code}")
    
    if response.status_code == 200:
        print("\n✓ 데이터 수신 성공!")
        print("\n[실제 데이터 샘플]")
        print("-" * 80)
        
        lines = response.text.strip().split('\n')
        print(f"총 {len(lines)}줄")
        
        # 처음 10줄만 출력
        for i, line in enumerate(lines[:10]):
            print(f"줄 {i+1}: {line}")
        
        print("-" * 80)
        
        # 데이터프레임으로 변환 시도
        if len(lines) >= 2:
            header = lines[0].split()
            data = []
            
            for line in lines[1:]:
                if line.strip():
                    data.append(line.split())
            
            if data:
                df = pd.DataFrame(data, columns=header)
                print(f"\n✓ 데이터프레임 생성 성공!")
                print(f"행 개수: {len(df)}")
                print(f"컬럼: {df.columns.tolist()}")
                
                print("\n[데이터 샘플]")
                print(df.head())
                
                # 일사량 관련 컬럼 찾기
                print("\n[일사량 관련 컬럼 찾기]")
                solar_keywords = ['SI', 'SUM', 'SOLAR', 'RAD', 'SS', 'solar', 'rad']
                found_cols = []
                
                for col in df.columns:
                    for keyword in solar_keywords:
                        if keyword.lower() in col.lower():
                            found_cols.append(col)
                            break
                
                if found_cols:
                    print(f"✓ 일사량 관련 컬럼 발견: {found_cols}")
                    
                    for col in found_cols:
                        print(f"\n[{col} 컬럼 샘플 데이터]")
                        print(df[col].head(10))
                else:
                    print("✗ 일사량 관련 컬럼을 찾을 수 없습니다.")
                    print("모든 컬럼 목록:")
                    for i, col in enumerate(df.columns, 1):
                        print(f"  {i}. {col}")
                
            else:
                print("\n✗ 데이터 행이 없습니다.")
        else:
            print("\n✗ 헤더만 있고 데이터가 없습니다.")
            
    else:
        print(f"\n✗ API 오류: {response.status_code}")
        print(f"응답: {response.text[:500]}")
        
except Exception as e:
    print(f"\n✗ 요청 실패: {e}")

print("\n" + "=" * 80)
print("테스트 완료!")
print("=" * 80)
print("\n다음 단계:")
print("1. 위 결과에서 일사량 컬럼명을 확인하세요")
print("2. 해당 컬럼명을 메인 코드에 적용하세요")
print("3. 데이터가 없다면 다른 지점(stn)이나 기간을 시도해보세요")

기상청 API 응답 구조 확인

[Step 1] help=1로 API 구조 확인
요청 URL: https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php
파라미터: {'tm1': '202101010000', 'tm2': '202101011000', 'stn': '108', 'authKey': '6XEt4XIFQJCxLeFyBYCQjg', 'help': 1}
--------------------------------------------------------------------------------

응답 상태 코드: 200

✓ API 연결 성공!

[API 응답 내용]
--------------------------------------------------------------------------------
#START7777
#--------------------------------------------------------------------------------------------------
#  기상청 지상관측 시간자료 [입력인수형태][예] ?tm=201007151200&stn=0&help=1
#--------------------------------------------------------------------------------------------------
#  1. TM     : 관측시각 (KST)
#  2. STN    : 국내 지점번호
#  3. WD     : 풍향 (36방위)
#  4. WS     : 풍속 (m/s)
#  5. GST_WD : 돌풍향 (36방위)
#  6. GST_WS : 돌풍속 (m/s)
#  7. GST_TM : 돌풍속이 관측된 시각 (시분)
#  8. PA     : 현지기압 (hPa)
#  9. PS     : 해면기압 (hPa)
# 10. PT     : 기압변화경향 (Code 0200) 
# 11. PR     : 기압변화량 (hPa)
# 12. TA 

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

# API 설정
AUTH_KEY = '6XEt4XIFQJCxLeFyBYCQjg' 
BASE_URL = 'https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php'

STATIONS = {'서울': '108', '울산': '152'}

def get_monthly_weather_totals(year, month, station_code, auth_key):
    """특정 월의 일조시간 합계와 일사량 합계를 가져옴"""
    start_date = datetime(year, month, 1)
    if month == 12:
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        end_date = datetime(year, month + 1, 1) - timedelta(days=1)
    
    tm1 = start_date.strftime('%Y%m%d0000')
    tm2 = end_date.strftime('%Y%m%d2300')
    
    params = {'tm1': tm1, 'tm2': tm2, 'stn': station_code, 'authKey': auth_key, 'help': 0}
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        lines = response.text.strip().split('\n')
        data_lines = [line for line in lines if not line.startswith('#') and line.strip()]
        
        if not data_lines: return None, None
        
        df = pd.DataFrame([line.split() for line in data_lines])
        
        # 컬럼 인덱스: 29(SS-일조시간), 30(SI-일사량)
        ss_values = pd.to_numeric(df.iloc[:, 29], errors='coerce')
        si_values = pd.to_numeric(df.iloc[:, 30], errors='coerce')
        
        # [핵심] 결측치 제외하고 한 달치 모두 더하기 (합계)
        ss_total = ss_values[ss_values >= 0].sum()
        si_total = si_values[si_values >= 0].sum()
        
        return ss_total, si_total
    except:
        return None, None

# 데이터 수집 실행
all_weather = []
for region in ['서울', '울산']:
    print(f"\n[{region}] 데이터 수집 중...")
    for year in range(2021, 2025):
        for month in range(1, 13):
            if year == 2024 and month > 12: break # 현재 시점까지
            
            ss_sum, si_sum = get_monthly_weather_totals(year, month, STATIONS[region], AUTH_KEY)
            if ss_sum is not None:
                all_weather.append({
                    '지역': region, '연도': year, '월': month,
                    '월합계일조(hr)': round(ss_sum, 2),
                    '월합계일사(MJ/m²)': round(si_sum, 2)
                })
                print(f"  {year}-{month:02d}: 일조 {ss_sum:.1f}hr, 일사 {si_sum:.1f}MJ", end='\r')
            time.sleep(0.2)

df_weather = pd.DataFrame(all_weather)
df_weather.to_csv('weather_totals.csv', index=False, encoding='utf-8-sig')
print("\n✅ 수집 완료 및 저장됨")



[서울] 데이터 수집 중...
  2024-12: 일조 223.0hr, 일사 108.0MJ
[울산] 데이터 수집 중...
  2024-12: 일조 220.0hr, 일사 61.0MJJ
✅ 수집 완료 및 저장됨


In [ ]:
import requests
import pandas as pd
import numpy as np
import calendar
import time
import io

# =========================
# 설정
# =========================
AUTH_KEY = "6XEt4XIFQJCxLeFyBYCQjg"  # 너 apihub 인증키
BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php"

STATIONS = {"서울": "108", "울산": "152"}

# sfctm3 헤더에서 찾을 변수 코드 후보들
CANDIDATES = {
    "SS": ["SS"],                      # 일조(시간자료) -> 월합계
    "SI": ["SI"],                      # 일사(시간자료) -> 월합계
    "TA": ["TA", "TA_AVG"],            # 기온 -> 월평균
    "HM": ["HM", "HM_AVG"],            # 습도 -> 월평균
    "RN": ["RN", "RNS", "RA"],         # 강수 -> 월합계 (있으면)
    "CA": ["CA", "CA_TOT", "CA_AVG"],  # 운량 -> 월평균 (있으면)
}

# =========================
# 내부 함수들
# =========================
def _find_header_tokens(text: str):
    """'# YYMMDDHHMI STN ...' 헤더 라인을 찾아 토큰 리스트로 반환"""
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("#") and "YYMMDDHHMI" in line and "STN" in line:
            return line.lstrip("#").strip().split()
    raise ValueError("헤더 라인('# YYMMDDHHMI STN ...')을 찾지 못했습니다. (help=1로 호출되는지 확인)")

def _pick_col_index(header_tokens, candidates):
    """header_tokens에서 candidates 중 존재하는 첫 번째의 인덱스 반환"""
    for c in candidates:
        if c in header_tokens:
            return header_tokens.index(c)
    return None

def _read_data_df(text: str) -> pd.DataFrame:
    """주석(#) 제거 후 공백 구분으로 데이터 로드"""
    df = pd.read_csv(io.StringIO(text), sep=r"\s+", comment="#", header=None)
    df = df.dropna(how="all")
    return df

def get_monthly_weather_totals_sfctm3(year, month, station_code, auth_key, sleep_sec=0.15):
    """
    sfctm3 시간자료로부터 월 집계:
    - SS, SI: 월합계
    - TA, HM, CA: 월평균
    - RN: 월합계(있으면)
    """
    last_day = calendar.monthrange(year, month)[1]
    tm1 = f"{year}{month:02d}01" + "0000"
    tm2 = f"{year}{month:02d}{last_day:02d}" + "2300"

    params = {
        "tm1": tm1,
        "tm2": tm2,
        "stn": station_code,
        "authKey": auth_key,
        "help": 1,   # ✅ 헤더 포함시키기(중요)
    }

    r = requests.get(BASE_URL, params=params, timeout=30)

    if r.status_code != 200:
        return {"error": f"HTTP {r.status_code}", "detail": r.text[:300]}

    text = r.text

    try:
        header = _find_header_tokens(text)
    except Exception as e:
        return {"error": "HEADER_NOT_FOUND", "detail": str(e) + " / " + text[:300]}

    df = _read_data_df(text)
    if df.empty:
        return None

    # 컬럼 인덱스 찾기
    idx = {}
    for k, cand in CANDIDATES.items():
        idx[k] = _pick_col_index(header, cand)

    def col_series(code):
        i = idx.get(code)
        if i is None or i >= df.shape[1]:
            return pd.Series(dtype=float)
        return pd.to_numeric(df.iloc[:, i], errors="coerce")

    ss = col_series("SS")
    si = col_series("SI")
    ta = col_series("TA")
    hm = col_series("HM")
    rn = col_series("RN")
    ca = col_series("CA")

    out = {
        "연도": year,
        "월": month,
        "월합계일조(hr)": float(ss.where(ss >= 0).sum()) if not ss.empty else np.nan,
        "월합계일사(MJ/m²)": float(si.where(si >= 0).sum()) if not si.empty else np.nan,
        "평균 기온(°C)": float(ta.where(ta > -900).mean()) if not ta.empty else np.nan,
        "평균 습도(%)": float(hm.where(hm >= 0).mean()) if not hm.empty else np.nan,
        "월합계강수량(mm)": float(rn.where(rn >= 0).sum()) if not rn.empty else np.nan,
        "평균 운량(1/10)": float(ca.where(ca >= 0).mean()) if not ca.empty else np.nan,
    }

    time.sleep(sleep_sec)
    return out

# =========================
# 실행: 2021~2024 서울/울산 월자료 생성
# =========================
all_weather = []

for region, stn in STATIONS.items():
    print(f"\n[{region}] sfctm3 기반 월집계 수집 중...")
    for year in range(2021, 2025):
        for month in range(1, 13):
            row = get_monthly_weather_totals_sfctm3(year, month, stn, AUTH_KEY)

            if row is None:
                continue

            if isinstance(row, dict) and "error" in row:
                print(f"\n⚠️ {region} {year}-{month:02d} 에러: {row['error']} / {row.get('detail','')}")
                continue

            row["지역"] = region
            all_weather.append(row)

            print(
                f"  {year}-{month:02d}: "
                f"일조 {row['월합계일조(hr)']:.1f}h, "
                f"일사 {row['월합계일사(MJ/m²)']:.1f}MJ",
                end="\r"
            )

df_weather = pd.DataFrame(all_weather)

# (선택) 비정상 필터: 월합계일조 400h 이상 제거
df_weather_clean = df_weather[df_weather["월합계일조(hr)"] < 400].copy()

df_weather_clean = df_weather_clean[
    ["지역", "연도", "월",
     "월합계일조(hr)", "월합계일사(MJ/m²)",
     "평균 습도(%)", "평균 기온(°C)", "월합계강수량(mm)", "평균 운량(1/10)"]
]

df_weather_clean.to_csv("weather_totals_temp.csv", index=False, encoding="utf-8-sig")
print("\n✅ 저장 완료: weather_totals_temp.csv")



[서울] sfctm3 기반 월집계 수집 중...
  2024-12: 일조 206.3h, 일사 288.5MJ
[울산] sfctm3 기반 월집계 수집 중...
  2024-12: 일조 236.0h, 일사 329.8MJ
✅ 저장 완료: weather_totals_temp.csv


In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
import zipfile


mpl.rcParams["font.family"] = "Malgun Gothic"   # ✅ 윈도우 기본 한글폰트
mpl.rcParams["axes.unicode_minus"] = False      # ✅ 마이너스(-) 깨짐 방지



# =========================
# 0) 파일 경로
# =========================
SOLAR_PATH = "U통합자료+누적설비용량+누적발전소개수.csv"  # 태양광 데이터
WEATHER_PATH = "weather_totals_temp.csv"                # sfctm3로 만든 기상 월집계

OUT_DIR = Path("corr_result_sfctm3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 1) 파일명 안전 처리 (Windows용 필수)
# =========================
def safe_filename(s: str, max_len: int = 140) -> str:
    s = str(s)

    # 경로로 해석되는 문자/윈도우 금지 문자 처리
    s = s.replace("/", "_")
    s = s.replace("\\", "_")
    s = s.replace(":", "_")
    s = s.replace("*", "_")
    s = s.replace("?", "")
    s = s.replace('"', "")
    s = s.replace("<", "_")
    s = s.replace(">", "_")
    s = s.replace("|", "_")
    s = s.replace("\n", " ").replace("\r", " ")

    # 공백/연속 언더스코어 정리
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"_+", "_", s)

    # 너무 길면 잘라내기(경로 길이 문제 방지)
    return s[:max_len]

# =========================
# 2) 유틸: CSV 인코딩 유연 로더
# =========================
def read_csv_flexible(path: str) -> pd.DataFrame:
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path)

# =========================
# 3) 데이터 로드
# =========================
df_solar = read_csv_flexible(SOLAR_PATH)
df_weather = read_csv_flexible(WEATHER_PATH)

# =========================
# 4) 기상 데이터 정제
# =========================
df_weather["지역"] = df_weather["지역"].astype(str).str.strip()
df_weather["연도"] = pd.to_numeric(df_weather["연도"], errors="coerce").astype("Int64")
df_weather["월"] = pd.to_numeric(df_weather["월"], errors="coerce").astype("Int64")

weather_cols = [
    "월합계일조(hr)",
    "월합계일사(MJ/m²)",     # <- 파일명에 / 들어가므로 반드시 safe_filename 필요
    "평균 습도(%)",
    "평균 기온(°C)",
    "월합계강수량(mm)",
    "평균 운량(1/10)",       # <- 파일명에 / 들어가므로 반드시 safe_filename 필요
]
for c in weather_cols:
    if c in df_weather.columns:
        df_weather[c] = pd.to_numeric(df_weather[c], errors="coerce")

# 비정상 제거: 한달 일조 400h 초과 제거(기존 로직 유지)
df_weather_clean = df_weather[df_weather["월합계일조(hr)"] < 400].copy()

# =========================
# 5) 태양광 데이터 전처리 (melt)
# =========================
df_solar["누적 설비용량(MW)"] = (
    df_solar["누적 설비용량(MW)"].astype(str).str.replace(",", "", regex=False)
)
df_solar["누적 설비용량(MW)"] = pd.to_numeric(df_solar["누적 설비용량(MW)"], errors="coerce")

month_cols = [f"{i}월" for i in range(1, 13)]
df_solar_melted = df_solar.melt(
    id_vars=["연도", "구분", "누적 설비용량(MW)"],
    value_vars=month_cols,
    var_name="월",
    value_name="발전량"
)

df_solar_melted["월"] = df_solar_melted["월"].str.replace("월", "", regex=False).astype(int)
df_solar_melted["연도"] = pd.to_numeric(df_solar_melted["연도"], errors="coerce").astype("Int64")
df_solar_melted["구분"] = df_solar_melted["구분"].astype(str).str.strip()

df_solar_melted["발전량"] = (
    df_solar_melted["발전량"].astype(str).str.replace(",", "", regex=False)
)
df_solar_melted["발전량"] = pd.to_numeric(df_solar_melted["발전량"], errors="coerce")

df_solar_melted = df_solar_melted.dropna(subset=["연도", "월", "구분", "발전량", "누적 설비용량(MW)"])

# =========================
# 6) 병합
# =========================
df_merged = pd.merge(
    df_solar_melted,
    df_weather_clean,
    left_on=["구분", "연도", "월"],
    right_on=["지역", "연도", "월"],
    how="inner"
)

# =========================
# 7) 이용률 계산 (윤년 포함 자동)
# =========================
period = pd.PeriodIndex(year=df_merged["연도"].astype(int), month=df_merged["월"].astype(int), freq="M")
df_merged["일수"] = period.days_in_month

df_merged["이용률"] = (df_merged["발전량"] / (df_merged["누적 설비용량(MW)"] * 24 * df_merged["일수"])) * 100

merged_out = OUT_DIR / "solar_weather_merged_sfctm3.csv"
df_merged.to_csv(merged_out, index=False, encoding="utf-8-sig")

# =========================
# 8) 상관분석 + 히트맵 + 산점도/회귀선
# =========================
analysis_cols = ["이용률"] + weather_cols

def corr_heatmap(corr: pd.DataFrame, title: str, out_file: Path):
    fig = plt.figure(figsize=(8.5, 6.5), dpi=170)
    ax = fig.add_subplot(111)
    data = corr.values.astype(float)

    im = ax.imshow(data)  # 색 지정 안함(기본)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)
    ax.set_title(title)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            ax.text(j, i, f"{v:.2f}" if np.isfinite(v) else "NaN",
                    ha="center", va="center", fontsize=8)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_file, bbox_inches="tight")
    plt.close(fig)

def scatter_with_regline(sub: pd.DataFrame, x: str, y: str, title: str, out_file: Path):
    d = sub[[x, y]].dropna()
    fig = plt.figure(figsize=(6.5, 5), dpi=170)
    ax = fig.add_subplot(111)

    ax.scatter(d[x], d[y], alpha=0.8)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{title}\n(n={len(d)})")

    if len(d) >= 2 and d[x].nunique() >= 2:
        m, b = np.polyfit(d[x].values, d[y].values, 1)
        xs = np.linspace(d[x].min(), d[x].max(), 100)
        ys = m * xs + b
        ax.plot(xs, ys, linewidth=2)

        yhat = m * d[x].values + b
        ss_res = np.sum((d[y].values - yhat) ** 2)
        ss_tot = np.sum((d[y].values - np.mean(d[y].values)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
        ax.text(0.02, 0.98, f"y={m:.3f}x+{b:.3f}\n$R^2$={r2:.3f}",
                transform=ax.transAxes, ha="left", va="top", fontsize=9)

    fig.tight_layout()
    fig.savefig(out_file, bbox_inches="tight")
    plt.close(fig)

# 폴더 준비
viz_dir = OUT_DIR / "viz"
viz_dir.mkdir(parents=True, exist_ok=True)

# 전체(서울+울산) Pearson/Spearman
corr_pearson_all = df_merged[analysis_cols].corr(method="pearson", numeric_only=True)
corr_spearman_all = df_merged[analysis_cols].corr(method="spearman", numeric_only=True)

corr_pearson_all.to_csv(OUT_DIR / "corr_pearson_all.csv", encoding="utf-8-sig")
corr_spearman_all.to_csv(OUT_DIR / "corr_spearman_all.csv", encoding="utf-8-sig")

corr_heatmap(corr_pearson_all, "[전체] Pearson 상관행렬", viz_dir / "heatmap_pearson_all.png")
corr_heatmap(corr_spearman_all, "[전체] Spearman 상관행렬", viz_dir / "heatmap_spearman_all.png")

# 지역별
regions = sorted(df_merged["지역"].dropna().unique().tolist())
for region in regions:
    sub = df_merged[df_merged["지역"] == region].copy()

    corr_p = sub[analysis_cols].corr(method="pearson", numeric_only=True)
    corr_s = sub[analysis_cols].corr(method="spearman", numeric_only=True)

    corr_p.to_csv(OUT_DIR / f"corr_pearson_{safe_filename(region)}.csv", encoding="utf-8-sig")
    corr_s.to_csv(OUT_DIR / f"corr_spearman_{safe_filename(region)}.csv", encoding="utf-8-sig")

    corr_heatmap(corr_p, f"[{region}] Pearson 상관행렬",
                 viz_dir / f"heatmap_pearson_{safe_filename(region)}.png")
    corr_heatmap(corr_s, f"[{region}] Spearman 상관행렬",
                 viz_dir / f"heatmap_spearman_{safe_filename(region)}.png")

# 산점도 + 회귀선: 이용률 vs 각 기상변수
scatter_dir = viz_dir / "scatter_reg"
scatter_dir.mkdir(parents=True, exist_ok=True)

for region in ["전체"] + regions:
    sub = df_merged if region == "전체" else df_merged[df_merged["지역"] == region].copy()
    for x in weather_cols:
        out = scatter_dir / f"scatter_{safe_filename(region)}_이용률_vs_{safe_filename(x)}.png"
        scatter_with_regline(
            sub,
            x=x,
            y="이용률",
            title=f"[{region}] 이용률 vs {x}",
            out_file=out
        )

# 요약표(이용률과 각 변수의 Pearson/Spearman만)
rows = []
for region in ["전체"] + regions:
    sub = df_merged if region == "전체" else df_merged[df_merged["지역"] == region].copy()
    cp = sub[analysis_cols].corr(method="pearson", numeric_only=True)
    cs = sub[analysis_cols].corr(method="spearman", numeric_only=True)
    for w in weather_cols:
        rows.append({
            "지역": region,
            "변수": w,
            "Pearson(이용률-변수)": float(cp.loc["이용률", w]),
            "Spearman(이용률-변수)": float(cs.loc["이용률", w]),
            "표본수(개월)": int(sub[["이용률", w]].dropna().shape[0]),
        })

summary = pd.DataFrame(rows)
summary_path = OUT_DIR / "corr_summary_utilization.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

# 결과 zip 패키징
zip_path = OUT_DIR / "corr_result_sfctm3_package.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(OUT_DIR)))

print("✅ 완료!")
print("병합 데이터:", merged_out)
print("요약표:", summary_path)
print("히트맵/산점도 폴더:", viz_dir)
print("전체 패키지(zip):", zip_path)

# (옵션) 콘솔 출력: 지역별 Spearman 상관행렬
for region in regions:
    sub = df_merged[df_merged["지역"] == region]
    corr_s = sub[analysis_cols].corr(method="spearman", numeric_only=True)
    print(f"\n[{region}] Spearman 상관행렬")
    print(corr_s.round(4))


C:\Users\bdy00\AppData\Local\Temp\ipykernel_6116\3692095787.py:130: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  period = pd.PeriodIndex(year=df_merged["연도"].astype(int), month=df_merged["월"].astype(int), freq="M")


✅ 완료!
병합 데이터: corr_result_sfctm3\solar_weather_merged_sfctm3.csv
요약표: corr_result_sfctm3\corr_summary_utilization.csv
히트맵/산점도 폴더: corr_result_sfctm3\viz
전체 패키지(zip): corr_result_sfctm3\corr_result_sfctm3_package.zip

[서울] Spearman 상관행렬
                 이용률  월합계일조(hr)  월합계일사(MJ/m²)  평균 습도(%)  평균 기온(°C)  \
이용률           1.0000     0.5849        0.9348   -0.2016     0.4718   
월합계일조(hr)     0.5849     1.0000        0.4515   -0.7149    -0.2269   
월합계일사(MJ/m²)  0.9348     0.4515        1.0000    0.0165     0.6592   
평균 습도(%)     -0.2016    -0.7149        0.0165    1.0000     0.6785   
평균 기온(°C)     0.4718    -0.2269        0.6592    0.6785     1.0000   
월합계강수량(mm)    0.1901    -0.4059        0.4095    0.7661     0.7756   
평균 운량(1/10)   0.2067    -0.5429        0.4066    0.7825     0.7916   

              월합계강수량(mm)  평균 운량(1/10)  
이용률               0.1901       0.2067  
월합계일조(hr)        -0.4059      -0.5429  
월합계일사(MJ/m²)      0.4095       0.4066  
평균 습도(%)          0.7661       0.7825  
평균 

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

# =========================
# 0) 한글 폰트 (Windows)
# =========================
mpl.rcParams["font.family"] = "Malgun Gothic"
mpl.rcParams["axes.unicode_minus"] = False

# =========================
# 1) 설정
# =========================
IN_PATH = "corr_result_sfctm3/solar_weather_merged_sfctm3.csv"  # 병합파일(이용률 포함)
OUT_DIR = Path("corr_anomaly_result")
OUT_DIR.mkdir(parents=True, exist_ok=True)

cols = [
    "이용률",
    "월합계일조(hr)",
    "월합계일사(MJ/m²)",
    "평균 습도(%)",
    "평균 기온(°C)",
    "월합계강수량(mm)",
    "평균 운량(1/10)",
]

def safe_filename(s: str, max_len: int = 140) -> str:
    s = str(s)
    s = s.replace("/", "_").replace("\\", "_")
    s = s.replace(":", "_").replace("*", "_").replace("?", "")
    s = s.replace('"', "").replace("<", "_").replace(">", "_").replace("|", "_")
    s = s.replace("\n", " ").replace("\r", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"_+", "_", s)
    return s[:max_len]

def corr_heatmap(corr: pd.DataFrame, title: str, out_file: Path):
    fig = plt.figure(figsize=(9, 7), dpi=180)
    ax = fig.add_subplot(111)
    data = corr.values.astype(float)

    im = ax.imshow(data)

    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)
    ax.set_title(title)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            ax.text(j, i, f"{v:.2f}" if np.isfinite(v) else "NaN",
                    ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_file, bbox_inches="tight")
    plt.close(fig)

# =========================
# 2) 데이터 로드
# =========================
df = pd.read_csv(IN_PATH, encoding="utf-8-sig")

# 숫자 컬럼 강제 변환
for c in cols + ["연도", "월"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["지역"] = df["지역"].astype(str).str.strip()

# 분석에 필요한 최소 결측 제거
df_base = df[["지역", "연도", "월"] + cols].dropna(subset=["지역", "연도", "월"]).copy()

# =========================
# 3) (원본) Spearman 상관
# =========================
corr_raw_all = df_base[cols].corr(method="spearman")
corr_raw_all.to_csv(OUT_DIR / "corr_spearman_raw_all.csv", encoding="utf-8-sig")
corr_heatmap(corr_raw_all, "[전체] Spearman (원본값)", OUT_DIR / "heatmap_spearman_raw_all.png")

# 지역별 원본
regions = sorted(df_base["지역"].dropna().unique().tolist())
for r in regions:
    sub = df_base[df_base["지역"] == r]
    corr_r = sub[cols].corr(method="spearman")
    corr_r.to_csv(OUT_DIR / f"corr_spearman_raw_{safe_filename(r)}.csv", encoding="utf-8-sig")
    corr_heatmap(corr_r, f"[{r}] Spearman (원본값)", OUT_DIR / f"heatmap_spearman_raw_{safe_filename(r)}.png")

# =========================
# 4) (anomaly) 계절성 제거: (지역,월) 평균을 뺌
# =========================
df_anom = df_base.copy()

for c in cols:
    # ✅ 같은 지역 + 같은 월의 평균(2021~2024)을 구해서 뺌
    month_mean = df_anom.groupby(["지역", "월"])[c].transform("mean")
    df_anom[c + "_anom"] = df_anom[c] - month_mean

anom_cols = [c + "_anom" for c in cols]

# anomaly 데이터 저장
anom_out = OUT_DIR / "solar_weather_anomaly.csv"
df_anom[["지역", "연도", "월"] + anom_cols].to_csv(anom_out, index=False, encoding="utf-8-sig")

# anomaly 상관(전체)
corr_anom_all = df_anom[anom_cols].corr(method="spearman")
corr_anom_all.to_csv(OUT_DIR / "corr_spearman_anomaly_all.csv", encoding="utf-8-sig")
corr_heatmap(corr_anom_all, "[전체] Spearman (anomaly: 계절성 제거)", OUT_DIR / "heatmap_spearman_anomaly_all.png")

# 지역별 anomaly
for r in regions:
    sub = df_anom[df_anom["지역"] == r]
    corr_r = sub[anom_cols].corr(method="spearman")
    corr_r.to_csv(OUT_DIR / f"corr_spearman_anomaly_{safe_filename(r)}.csv", encoding="utf-8-sig")
    corr_heatmap(corr_r, f"[{r}] Spearman (anomaly: 계절성 제거)", OUT_DIR / f"heatmap_spearman_anomaly_{safe_filename(r)}.png")

# =========================
# 5) 비교표: 이용률 vs 각 변수 (원본 vs anomaly)
# =========================
rows = []
for r in ["전체"] + regions:
    sub_raw = df_base if r == "전체" else df_base[df_base["지역"] == r]
    sub_anm = df_anom if r == "전체" else df_anom[df_anom["지역"] == r]

    corr_raw = sub_raw[cols].corr(method="spearman")
    corr_anm = sub_anm[anom_cols].corr(method="spearman")

    for w in cols:
        if w == "이용률":
            continue
        rows.append({
            "지역": r,
            "변수": w,
            "Spearman(원본) 이용률-변수": float(corr_raw.loc["이용률", w]),
            "Spearman(anomaly) 이용률-변수": float(corr_anm.loc["이용률_anom", w + "_anom"]),
            "표본수(원본)": int(sub_raw[["이용률", w]].dropna().shape[0]),
            "표본수(anomaly)": int(sub_anm[["이용률_anom", w + "_anom"]].dropna().shape[0]),
        })

compare_df = pd.DataFrame(rows)
compare_df.to_csv(OUT_DIR / "compare_utilization_raw_vs_anomaly.csv", index=False, encoding="utf-8-sig")

print("✅ 완료!")
print("출력 폴더:", OUT_DIR.resolve())
print("anomaly CSV:", anom_out)
print("비교표 CSV:", (OUT_DIR / "compare_utilization_raw_vs_anomaly.csv"))
print("원본 히트맵:", (OUT_DIR / "heatmap_spearman_raw_all.png"))
print("anomaly 히트맵:", (OUT_DIR / "heatmap_spearman_anomaly_all.png"))



✅ 완료!
출력 폴더: C:\Users\bdy00\Desktop\KDT_RE_5th_Project1\Project\corr_anomaly_result
anomaly CSV: corr_anomaly_result\solar_weather_anomaly.csv
비교표 CSV: corr_anomaly_result\compare_utilization_raw_vs_anomaly.csv
원본 히트맵: corr_anomaly_result\heatmap_spearman_raw_all.png
anomaly 히트맵: corr_anomaly_result\heatmap_spearman_anomaly_all.png


In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path

# =========================
# 0) 한글 폰트(Windows)
# =========================
mpl.rcParams["font.family"] = "Malgun Gothic"
mpl.rcParams["axes.unicode_minus"] = False

# =========================
# 1) 입력/출력
# =========================
IN_PATH = "corr_result_sfctm3/solar_weather_merged_sfctm3.csv"  # 병합+이용률 파일
OUT_DIR = Path("corr_monthly_anomaly")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 분석 변수
cols = [
    "이용률",
    "월합계일조(hr)",
    "월합계일사(MJ/m²)",
    "평균 습도(%)",
    "평균 기온(°C)",
    "월합계강수량(mm)",
    "평균 운량(1/10)",
]

def safe_filename(s: str, max_len: int = 140) -> str:
    s = str(s)
    s = s.replace("/", "_").replace("\\", "_")
    s = s.replace(":", "_").replace("*", "_").replace("?", "")
    s = s.replace('"', "").replace("<", "_").replace(">", "_").replace("|", "_")
    s = s.replace("\n", " ").replace("\r", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"_+", "_", s)
    return s[:max_len]

def corr_heatmap(corr: pd.DataFrame, title: str, out_file: Path):
    fig = plt.figure(figsize=(9, 7), dpi=180)
    ax = fig.add_subplot(111)

    data = corr.values.astype(float)
    im = ax.imshow(data)

    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)
    ax.set_title(title)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            ax.text(j, i, f"{v:.2f}" if np.isfinite(v) else "NaN",
                    ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_file, bbox_inches="tight")
    plt.close(fig)

# =========================
# 2) 로드 + anomaly 생성(지역,월 평균 제거)
# =========================
df = pd.read_csv(IN_PATH, encoding="utf-8-sig")
df["지역"] = df["지역"].astype(str).str.strip()

# 숫자화
for c in cols + ["연도", "월"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["지역", "연도", "월"]).copy()

# anomaly 생성: (지역, 월) 평균을 빼서 계절성 제거
for c in cols:
    mean_rm = df.groupby(["지역", "월"])[c].transform("mean")
    df[c + "_anom"] = df[c] - mean_rm

anom_cols = [c + "_anom" for c in cols]

# =========================
# 3) 월별 Spearman 상관(2021~2024 통합)
#    - 전체(서울+울산) + 서울 + 울산 각각 저장
# =========================
groups = ["전체"] + sorted(df["지역"].unique().tolist())

# 요약표: 이용률과 각 변수 상관(월별)
summary_rows = []

for m in range(1, 13):
    for g in groups:
        sub = df[df["월"] == m].copy() if g == "전체" else df[(df["월"] == m) & (df["지역"] == g)].copy()

        # 표본 너무 적으면 스킵 (상관행렬이 불안정)
        if sub.shape[0] < 6:
            continue

        corr = sub[anom_cols].corr(method="spearman")

        # 저장
        name_tag = f"{g}_월{m:02d}"
        corr.to_csv(OUT_DIR / f"corr_spearman_anom_{safe_filename(name_tag)}.csv", encoding="utf-8-sig")
        corr_heatmap(
            corr,
            title=f"[{g}] {m}월 Spearman (anomaly, 2021~2024 통합)",
            out_file=OUT_DIR / f"heatmap_spearman_anom_{safe_filename(name_tag)}.png"
        )

        # 이용률 vs 각 변수만 요약(월별)
        for w in cols:
            if w == "이용률":
                continue
            x = "이용률_anom"
            y = w + "_anom"
            n = sub[[x, y]].dropna().shape[0]
            if n < 6:
                continue
            rho = sub[[x, y]].corr(method="spearman").iloc[0, 1]
            summary_rows.append({
                "그룹": g,
                "월": m,
                "변수": w,
                "Spearman(이용률_anom-변수_anom)": float(rho),
                "표본수": int(n)
            })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "summary_utilization_vs_weather_by_month_anomaly.csv", index=False, encoding="utf-8-sig")

print("✅ 완료!")
print("출력 폴더:", OUT_DIR.resolve())
print("월별 요약표:", (OUT_DIR / "summary_utilization_vs_weather_by_month_anomaly.csv"))


✅ 완료!
출력 폴더: C:\Users\bdy00\Desktop\KDT_RE_5th_Project1\Project\corr_monthly_anomaly
월별 요약표: corr_monthly_anomaly\summary_utilization_vs_weather_by_month_anomaly.csv


In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager as fm
from pathlib import Path

# =========================
# 0) 한글 폰트 설정 (Windows에서 글자 깨짐 방지)
# =========================
def set_korean_font():
    candidates = ["Malgun Gothic", "NanumGothic", "AppleGothic"]
    available = {f.name for f in fm.fontManager.ttflist}

    for name in candidates:
        if name in available:
            mpl.rcParams["font.family"] = name
            mpl.rcParams["axes.unicode_minus"] = False
            return

    # 마지막 방어: 윈도우 맑은고딕 ttf 직접 등록
    malgun_path = r"C:\Windows\Fonts\malgun.ttf"
    try:
        fm.fontManager.addfont(malgun_path)
        font_name = fm.FontProperties(fname=malgun_path).get_name()
        mpl.rcParams["font.family"] = font_name
        mpl.rcParams["axes.unicode_minus"] = False
    except Exception:
        # 폰트 못 찾으면 그냥 기본 폰트(깨질 수 있음)
        mpl.rcParams["axes.unicode_minus"] = False

set_korean_font()

# =========================
# 1) 설정
# =========================
IN_PATH = "corr_result_sfctm3/solar_weather_merged_sfctm3.csv"  # 병합+이용률 파일
OUT_DIR = Path("monthly_heatmaps_by_region_anomaly")
OUT_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = ["서울", "울산"]

cols = [
    "이용률",
    "월합계일조(hr)",
    "월합계일사(MJ/m²)",
    "평균 습도(%)",
    "평균 기온(°C)",
    "월합계강수량(mm)",
    "평균 운량(1/10)",
]

def safe_filename(s: str, max_len: int = 140) -> str:
    s = str(s)
    s = s.replace("/", "_").replace("\\", "_")
    s = s.replace(":", "_").replace("*", "_").replace("?", "")
    s = s.replace('"', "").replace("<", "_").replace(">", "_").replace("|", "_")
    s = s.replace("\n", " ").replace("\r", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"_+", "_", s)
    return s[:max_len]

def corr_heatmap(corr: pd.DataFrame, title: str, out_file: Path, vmin=-1, vmax=1):
    fig = plt.figure(figsize=(9, 7), dpi=180)
    ax = fig.add_subplot(111)

    data = corr.values.astype(float)
    im = ax.imshow(data, vmin=vmin, vmax=vmax)

    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)
    ax.set_title(title)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            ax.text(j, i, f"{v:.2f}" if np.isfinite(v) else "NaN",
                    ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_file, bbox_inches="tight")
    plt.close(fig)

# =========================
# 2) 로드 + anomaly 생성(지역,월 평균 제거)
# =========================
df = pd.read_csv(IN_PATH, encoding="utf-8-sig")
df["지역"] = df["지역"].astype(str).str.strip()

# 숫자화
for c in cols + ["연도", "월"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["지역", "연도", "월"]).copy()

# ✅ 계절성 제거(anomaly): (지역,월) 평균을 빼기
for c in cols:
    month_mean = df.groupby(["지역", "월"])[c].transform("mean")
    df[c + "_anom"] = df[c] - month_mean

anom_cols = [c + "_anom" for c in cols]

# =========================
# 3) 서울/울산 각각 월(1~12) 상관행렬 히트맵 저장
# =========================
for region in REGIONS:
    region_dir = OUT_DIR / safe_filename(region)
    region_dir.mkdir(parents=True, exist_ok=True)

    for m in range(1, 13):
        sub = df[(df["지역"] == region) & (df["월"] == m)].copy()

        # 표본수(2021~2024면 보통 4개)라서 상관이 불안정할 수 있음.
        # 그래도 일단 계산/저장은 하되, 너무 적으면 경고만 출력
        n = sub.shape[0]
        if n < 4:
            print(f"⚠️ {region} {m}월 표본수 {n}개 (상관 신뢰도 낮음)")

        corr = sub[anom_cols].corr(method="spearman")

        # 저장 (CSV + PNG)
        csv_path = region_dir / f"corr_spearman_anom_{safe_filename(region)}_월{m:02d}.csv"
        png_path = region_dir / f"heatmap_spearman_anom_{safe_filename(region)}_월{m:02d}.png"

        corr.to_csv(csv_path, encoding="utf-8-sig")
        corr_heatmap(
            corr,
            title=f"[{region}] {m}월 Spearman 상관행렬 (anomaly, 2021~2024 통합)",
            out_file=png_path
        )

print("✅ 완료!")
print("저장 폴더:", OUT_DIR.resolve())
print("예시 파일:", (OUT_DIR / "서울").resolve() if (OUT_DIR / "서울").exists() else OUT_DIR.resolve())


✅ 완료!
저장 폴더: C:\Users\bdy00\Desktop\KDT_RE_5th_Project1\Project\monthly_heatmaps_by_region_anomaly
예시 파일: C:\Users\bdy00\Desktop\KDT_RE_5th_Project1\Project\monthly_heatmaps_by_region_anomaly\서울


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

DATA_PATH = "corr_result_sfctm3/solar_weather_merged_sfctm3.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

df["지역"] = df["지역"].astype(str).str.strip()
for c in ["연도","월","이용률",
          "월합계일조(hr)","월합계일사(MJ/m²)",
          "평균 습도(%)","평균 기온(°C)",
          "월합계강수량(mm)","평균 운량(1/10)"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["지역","연도","월","이용률"]).copy()
df["연월"] = df["연도"].astype(int).astype(str) + "-" + df["월"].astype(int).astype(str).str.zfill(2)

# ✅ 컬럼명 매핑(안전한 영문 변수명)
rename_map = {
    "이용률": "util",
    "월합계일조(hr)": "sunshine_hr",
    "월합계일사(MJ/m²)": "solar_mj",
    "평균 습도(%)": "hum",
    "평균 기온(°C)": "temp_c",
    "월합계강수량(mm)": "rain_mm",
    "평균 운량(1/10)": "cloud",
}
df = df.rename(columns=rename_map)

weather_vars = ["sunshine_hr","solar_mj","hum","temp_c","rain_mm","cloud"]

# anomaly(계절성 제거)
USE_ANOMALY = True
if USE_ANOMALY:
    for c in ["util"] + weather_vars:
        m = df.groupby(["지역","월"])[c].transform("mean")
        df[c + "_anom"] = df[c] - m
    y = "util_anom"
    X = [v + "_anom" for v in weather_vars]
else:
    y = "util"
    X = weather_vars

df_model = df.dropna(subset=[y] + X).copy()

# --------- 모델1: 지역FE + 연월FE ----------
base_formula = f"{y} ~ " + " + ".join(X) + " + C(지역) + C(연월)"
m_base = smf.ols(base_formula, data=df_model).fit(cov_type="HC3")
print("\n=== [모델1] FE(지역 + 연월) ===")
print(m_base.summary())

# --------- 모델2: FE + 지역×기상 상호작용(울산 효과 차이) ----------
# (울산과 서울 비교 목적이면 지역 2개일 때 특히 직관적)
key_vars = ["solar_mj_anom", "sunshine_hr_anom", "cloud_anom", "rain_mm_anom"]
inter_terms = " + ".join([f"C(지역):{kv}" for kv in key_vars])

inter_formula = base_formula + " + " + inter_terms
m_inter = smf.ols(inter_formula, data=df_model).fit(cov_type="HC3")
print("\n=== [모델2] FE + 지역×기상 상호작용 ===")
print(m_inter.summary())

# 예측/잔차 저장
df_model["pred_base"] = m_base.predict(df_model)
df_model["resid_base"] = df_model[y] - df_model["pred_base"]

df_model["pred_inter"] = m_inter.predict(df_model)
df_model["resid_inter"] = df_model[y] - df_model["pred_inter"]

df_model.to_csv("panel_reg_results_with_residuals.csv", index=False, encoding="utf-8-sig")
print("\n✅ 저장: panel_reg_results_with_residuals.csv")



=== [모델1] FE(지역 + 연월) ===
                            OLS Regression Results                            
Dep. Variable:              util_anom   R-squared:                       0.953
Model:                            OLS   Adj. R-squared:                  0.891
Method:                 Least Squares   F-statistic:                     280.0
Date:                Mon, 05 Jan 2026   Prob (F-statistic):           1.79e-40
Time:                        11:20:54   Log-Likelihood:                -7.6091
No. Observations:                  96   AIC:                             125.2
Df Residuals:                      41   BIC:                             266.3
Df Model:                          54                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept    

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# =========================
# (A) 경로 설정
# =========================
SOLAR_PATH = "U통합자료+누적설비용량+누적발전소개수.csv"
WEATHER_FEATURES_PATH = "weather_integrated_features.csv"  # 있으면 이걸 우선 사용
WEATHER_TOTALS_PATH = "weather_totals.csv"                 # 없으면 fallback

OUT_WEATHER_IMPUTED = "weather_imputed_monthmean.csv"
OUT_MERGED_IMPUTED = "solar_weather_merged_imputed.csv"
OUT_MONTHLY_MEAN = "monthly_climatology_2021_2024_seoul_ulsan_imputed.csv"
OUT_UTIL_PLOT = "utilization_by_month_seoul_ulsan_imputed.png"
OUT_GAP_PLOT = "util_gap_by_month_imputed.png"

# =========================
# (B) 태양광 데이터 로드/전처리
# =========================
df_solar = pd.read_csv(SOLAR_PATH, encoding="cp949")
df_solar["구분"] = df_solar["구분"].astype(str).str.strip()

# 누적 설비용량 숫자화
cap_col = "누적 설비용량(MW)"
df_solar[cap_col] = (
    df_solar[cap_col].astype(str).str.replace(",", "", regex=False).astype(float)
)

# 월 컬럼 펼치기
month_cols = [f"{i}월" for i in range(1, 13)]
df_solar_melt = df_solar.melt(
    id_vars=["연도", "구분", cap_col],
    value_vars=month_cols,
    var_name="월",
    value_name="발전량"
)
df_solar_melt["월"] = df_solar_melt["월"].str.replace("월", "", regex=False).astype(int)
df_solar_melt["발전량"] = pd.to_numeric(df_solar_melt["발전량"], errors="coerce")

# 이용률 계산 (윤년 2024-02만 29일)
days_map = {1:31, 2:28, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31}
df_solar_melt["일수"] = df_solar_melt.apply(
    lambda x: 29 if (int(x["연도"]) == 2024 and int(x["월"]) == 2) else days_map[int(x["월"])],
    axis=1
)
df_solar_melt["이용률"] = (
    df_solar_melt["발전량"] / (df_solar_melt[cap_col] * 24 * df_solar_melt["일수"])
) * 100

# 서울/울산만(원하면 주석처리)
df_solar_melt = df_solar_melt[df_solar_melt["구분"].isin(["서울", "울산"])].copy()

# =========================
# (C) 기상 데이터 로드 (features 우선, 없으면 totals)
# =========================
if Path(WEATHER_FEATURES_PATH).exists():
    df_w = pd.read_csv(WEATHER_FEATURES_PATH, encoding="utf-8-sig")
else:
    df_w = pd.read_csv(WEATHER_TOTALS_PATH, encoding="utf-8-sig")

df_w["지역"] = df_w["지역"].astype(str).str.strip()
df_w = df_w[df_w["지역"].isin(["서울", "울산"])].copy()

# 반드시 존재해야 하는 두 컬럼(월합계)
sun_col = "월합계일조(hr)"
sol_col = "월합계일사(MJ/m²)"

for c in ["연도", "월", sun_col, sol_col]:
    if c in df_w.columns:
        df_w[c] = pd.to_numeric(df_w[c], errors="coerce")

# =========================
# (D) 이상치 drop 대신 "월별 평균으로 대체(impute)"
#     - 조건: 월합계일조 >= 400 이면 비정상
# =========================
df_w["flag_abnormal_sunshine"] = df_w[sun_col] >= 400
df_w["flag_invalid_solar"] = df_w[sol_col].isna() | (df_w[sol_col] < 0)

# 대체 대상: 일조 이상치 or 결측 or 일사 결측/음수
df_w["flag_impute"] = df_w["flag_abnormal_sunshine"] | df_w[sun_col].isna() | df_w["flag_invalid_solar"]

# 대체 대상은 NaN으로 비워두고 나중에 월평균으로 채움
df_w.loc[df_w["flag_impute"], [sun_col, sol_col]] = np.nan

# (지역, 월) 기준 월평균 계산(정상 값들로만)
impute_table = (
    df_w.groupby(["지역", "월"])[[sun_col, sol_col]]
    .mean()
    .rename(columns={sun_col: f"{sun_col}_month_mean", sol_col: f"{sol_col}_month_mean"})
    .reset_index()
)

df_w2 = df_w.merge(impute_table, on=["지역", "월"], how="left")

# NaN인 곳을 (지역, 월) 평균으로 채움
df_w2[sun_col] = df_w2[sun_col].fillna(df_w2[f"{sun_col}_month_mean"])
df_w2[sol_col] = df_w2[sol_col].fillna(df_w2[f"{sol_col}_month_mean"])

# 저장(대체 완료 기상)
df_w2.drop(columns=[f"{sun_col}_month_mean", f"{sol_col}_month_mean"], errors="ignore") \
    .to_csv(OUT_WEATHER_IMPUTED, index=False, encoding="utf-8-sig")
print("✅ 저장:", OUT_WEATHER_IMPUTED)

# =========================
# (E) 태양광 + 기상 merge
# =========================
df_merged = pd.merge(
    df_solar_melt,
    df_w2,
    left_on=["구분", "연도", "월"],
    right_on=["지역", "연도", "월"],
    how="left"
)

df_merged.to_csv(OUT_MERGED_IMPUTED, index=False, encoding="utf-8-sig")
print("✅ 저장:", OUT_MERGED_IMPUTED)

# =========================
# (F) 2021~2024 통합 월평균(1~12월) 만들기
# =========================
vars_cols = ["이용률", sun_col, sol_col]
optional = ["평균 습도(%)", "평균 기온(°C)", "월합계강수량(mm)", "평균 운량(1/10)", "평균 강수량(mm)"]
for c in optional:
    if c in df_merged.columns and c not in vars_cols:
        vars_cols.append(c)

monthly_mean = df_merged.groupby(["지역", "월"])[vars_cols].mean().reset_index()
monthly_n = df_merged.groupby(["지역", "월"]).size().reset_index(name="n_years")
monthly_mean = monthly_mean.merge(monthly_n, on=["지역", "월"], how="left").sort_values(["지역", "월"])

monthly_mean.to_csv(OUT_MONTHLY_MEAN, index=False, encoding="utf-8-sig")
print("✅ 저장:", OUT_MONTHLY_MEAN)

# =========================
# (G) 월별 이용률 그래프 + 격차 그래프
# =========================
util_pv = monthly_mean.pivot(index="월", columns="지역", values="이용률").reset_index()

plt.figure()
plt.plot(util_pv["월"], util_pv["서울"], marker="o", label="Seoul")
plt.plot(util_pv["월"], util_pv["울산"], marker="o", label="Ulsan")
plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("Utilization (%)")
plt.title("Monthly average utilization (2021–2024 pooled, imputed weather)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_UTIL_PLOT, dpi=200)
plt.close()
print("✅ 저장:", OUT_UTIL_PLOT)

util_pv["gap_seoul_minus_ulsan"] = util_pv["서울"] - util_pv["울산"]
plt.figure()
plt.bar(util_pv["월"], util_pv["gap_seoul_minus_ulsan"])
plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("Seoul - Ulsan (pp)")
plt.title("Utilization gap by month (imputed weather)")
plt.tight_layout()
plt.savefig(OUT_GAP_PLOT, dpi=200)
plt.close()
print("✅ 저장:", OUT_GAP_PLOT)

print("\n끝! (drop 대신 impute 방식 적용 완료)")


✅ 저장: weather_imputed_monthmean.csv
✅ 저장: solar_weather_merged_imputed.csv
✅ 저장: monthly_climatology_2021_2024_seoul_ulsan_imputed.csv
✅ 저장: utilization_by_month_seoul_ulsan_imputed.png
✅ 저장: util_gap_by_month_imputed.png

끝! (drop 대신 impute 방식 적용 완료)
